In [ ]:
import polars as pl
from pathlib import Path
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import repeat

In [2]:
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

Folder where groups (data) folder should be: /data/users/bdupin/sam-honours


In [3]:
group = [
    "plants_plt_solanaceae"
]

In [6]:
custalo_cmd = ("clustalo "
               "-i {in_fasta} "
               "-o {out_fasta} "
               "--outfmt=fa "
               "-t DNA "
               "--full-iter "
               "--iter=3 "
               "--output-order=input-order "
               "--threads={cpus} "
               "--log={log_file} "
               "--verbose --force"
)

force_run = True
n_jobs = 32

In [7]:
def run_clustalo(gp, gp_dir, logs_dir):
    in_fasta = gp_dir / f"gp_{gp}.fasta"
    out_fasta = gp_dir / f"gp_{gp}_aligned.fasta"
    log_file = logs_dir / f"gp_{gp}_clustalo.log"

    if out_fasta.exists() and not force_run:
        return None  # skipped

    cmd = custalo_cmd.format(
        in_fasta=in_fasta, out_fasta=out_fasta, cpus=1, log_file=log_file
    ).split()
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if res.returncode != 0:
        (logs_dir / f"gp_{gp}_clustalo_error.log").write_text(res.stdout)
    return res.returncode

In [10]:
for g in group:
    print(f"Processing group: {g}")
    gdir = BASE / g

    gp_df = pl.read_csv(gdir / "gene_pairs_filtered.tsv", separator="\t")
    gp_dir = gdir / "gene_pair_fastas"
    logs_dir = gp_dir / "logs"
    logs_dir.mkdir(exist_ok=True)

    pairs = gp_df["gene_pair"].to_list()
    with ThreadPoolExecutor(max_workers=n_jobs) as ex:
        for gp, rc in zip(pairs, ex.map(run_clustalo, pairs, repeat(gp_dir), repeat(logs_dir))):
            print(f"{gp}: {'skipped' if rc is None else 'ok' if rc == 0 else f'error (exit {rc})'}")

Processing group: plants_plt_solanaceae
NDHA|NDHI|++: ok
NDHB|NDHB|--: ok
RPS15|NDHH|++: ok
TY|TE|--: ok
RPOA|RPS11|--: ok
TI|RNRS|--: ok
LHBA|TG|++: ok
PSBA|TK|--: ok
TV|RNRS|++: ok
RNRS|TI|++: ok
NDHK|NDHC|--: ok
RNRS|TV|--: ok
TS1|2|LHBA|-+: ok
TV|YCF15|-+: ok
CLPP|PSBT|-+: ok
RNR16|TI|++: ok
RPS14|PSAB|--: ok
PSBT|PSBN|+-: ok
TI|RNR16|--: ok
YCF15|TV|-+: ok
NDHE|PSAC|++: ok
RPS11|RPL36|--: ok
PSBN|PSBH|-+: ok
RPL22|RPS19|--: ok
PSBJ|PSBF|--: ok
TV|TM|-+: ok
PSBJ|PSBL|--: ok
TL1|2|CCSA|++: ok
NDHB|RPS7|--: ok
RPS7|NDHB|++: ok
TL1|2|NDHB|--: ok
PETB|PETB|++: ok
RPL32|SPRA|++: ok
TM|RPS14|--: ok
RPL14|RPL16|--: ok
NDHH|RPS15|--: ok
PSAC|NDHE|--: ok
TW|TP|--: ok
TR|ATPA|+-: ok
PETG|TW|+-: ok
RNR5|TR|++: ok
NDHE|NDHG|--: ok
RPS8|RPL14|--: ok
TN|RNR5|+-: ok
CCSA|TL1|2|--: ok
PETD|PETD|++: ok
RNR4.5|RNR23|--: ok
PETL|PETG|++: ok
PSAI|YCF4|++: ok
TR|RNR5|--: ok
TQ|PSBK|-+: ok
PSAC|NDHD|++: ok
NDHG|NDHE|++: ok
CEMA|PETA|++: ok
PSBI|TS1|2|+-: ok
NDHD|CCSA|+-: ok
RPS19|RPL2|--: ok
RPL36|INFA|

runtime: 55m27s
pairs: 195